# Visão da ABT final (camada Gold) — Home Credit Default Risk

Este notebook descreve a **ABT pronta para modelagem** (`Dados/Gold/abt.parquet`), gerada por
`abt_transform.py`. Diferente do `exp_analysis.ipynb` (que explora os **dados limpos** / Silver),
aqui olhamos a tabela **já transformada**: 1 linha por solicitação (`SK_ID_CURR`), todas as
variáveis **numéricas** (categóricas já passaram por Label Encoding), nulos imputados e enriquecida
com **28 agregações de histórico de crédito** (`BUREAU_*` de `bureau` + `PREV_*` de
`previous_application`).

In [ ]:
import sys, os
# Sobe para a raiz do projeto (paths de config.py são relativos a ela).
while not (os.path.isdir('Model') and os.path.isdir('Dados')):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

abt = pd.read_parquet('Dados/Gold/abt.parquet')
print(f"Shape da ABT: {abt.shape[0]:,} linhas x {abt.shape[1]} colunas")
print(f"Todas numéricas? {abt.select_dtypes('number').shape[1] == abt.shape[1]}")
abt.head()

## 1. Composição das colunas por bloco

In [ ]:
ID_COL, TARGET = 'SK_ID_CURR', 'TARGET'
bureau_cols = sorted([c for c in abt.columns if c.startswith('BUREAU_')])
prev_cols = sorted([c for c in abt.columns if c.startswith('PREV_')])
ratio_cols = [c for c in ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'ANNUITY_CREDIT_RATIO']
              if c in abt.columns]
app_cols = [c for c in abt.columns if c not in [ID_COL, TARGET] + bureau_cols + prev_cols + ratio_cols]

blocos = pd.DataFrame({
    'bloco': ['Identificador', 'Alvo', 'Application (solicitação)', 'Razões financeiras',
              'Histórico externo (BUREAU_*)', 'Histórico interno (PREV_*)'],
    'origem': ['-', '-', 'application_train', 'engenharia', 'bureau (agregado)', 'previous_application (agregado)'],
    'n_colunas': [1, 1, len(app_cols), len(ratio_cols), len(bureau_cols), len(prev_cols)],
})
print(f"Total de colunas: {abt.shape[1]} | Features (excl. ID/TARGET): {abt.shape[1]-2}")
display(blocos)

ax = sns.barplot(data=blocos, x='n_colunas', y='bloco', palette='viridis')
plt.title('Nº de colunas por bloco da ABT'); plt.xlabel('nº de colunas'); plt.ylabel('')
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y()+p.get_height()/2),
                ha='left', va='center', size=10)
plt.show()

> 📊 **Como interpretar:** a ABT tem **111 features**. A maioria vem da **solicitação** (`application`);
> **28** são **agregações de histórico de crédito** (`BUREAU_*` externo + `PREV_*` interno) e **3** são
> **razões financeiras** criadas por nós. O histórico foi o principal enriquecimento sobre a base original.

## 2. Cobertura do histórico (clientes com vs. sem registro)

In [ ]:
# Cliente sem histórico recebe 0 nas agregações (definido no abt_transform).
com_bureau = (abt['BUREAU_CREDIT_COUNT'] > 0).mean() if 'BUREAU_CREDIT_COUNT' in abt else np.nan
com_prev = (abt['PREV_APP_COUNT'] > 0).mean() if 'PREV_APP_COUNT' in abt else np.nan
print(f"Clientes COM histórico no bureau (externo):       {com_bureau:.1%}")
print(f"Clientes COM pedido anterior na Home Credit:      {com_prev:.1%}")

cov = pd.DataFrame({
    'fonte': ['Bureau (externo)', 'Previous (interno)'],
    'com_historico': [com_bureau, com_prev],
    'sem_historico': [1-com_bureau, 1-com_prev],
}).set_index('fonte')
cov.plot(kind='barh', stacked=True, color=['#2a9d8f', '#e76f51'])
plt.title('Cobertura do histórico de crédito na base'); plt.xlabel('proporção de clientes')
plt.legend(['com histórico', 'sem histórico'], loc='lower right'); plt.show()

> 📊 **Como interpretar:** **~86%** dos clientes têm histórico no **bureau** (externo) e **~95%** têm
> **pedido anterior** na Home Credit. Quem não tem recebe **0** nas agregações (tratado como "sem registro"),
> então a feature funciona para todos os clientes.

## 3. Correlação das novas features com o alvo (TARGET)

In [ ]:
hist_cols = bureau_cols + prev_cols
corr = abt[hist_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
print('Top 15 features de histórico por |correlação| com TARGET:')
top = corr.head(15)
display(top.to_frame('corr_com_TARGET'))

plt.figure(figsize=(9, 7))
sns.barplot(x=top.values, y=top.index, palette='coolwarm')
plt.title('Correlação das features de histórico com TARGET (top 15 por |valor|)')
plt.axvline(0, color='black', lw=0.8); plt.xlabel('correlação com TARGET')
plt.show()

> 📊 **Como interpretar:** entre as features de histórico, as de **dívida/atraso** (bureau) e **taxa de
> recusa** (previous) têm correlação **positiva** com a inadimplência (mais dívida/recusa → mais risco),
> enquanto **taxas de aprovação** têm correlação **negativa**. Coerente com o domínio de crédito.

## 4. Distribuição de features-chave por TARGET

In [ ]:
# Exemplos de variáveis de histórico com leitura de negócio direta.
exemplos = [c for c in ['BUREAU_DEBT_CREDIT_RATIO', 'BUREAU_ACTIVE_COUNT',
                        'PREV_REFUSED_RATE', 'PREV_APPROVAL_RATE'] if c in abt.columns]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, col in zip(axes.ravel(), exemplos):
    media = abt.groupby(TARGET)[col].mean()
    sns.barplot(x=media.index, y=media.values, palette='coolwarm', ax=ax)
    ax.set_title(f'Média de {col} por TARGET')
    ax.set_xlabel('TARGET (0=paga, 1=inadimplente)'); ax.set_ylabel(col)
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** comparando a **média por grupo**, os inadimplentes (`TARGET=1`) tendem a ter
> **mais créditos ativos** e **maior razão dívida/crédito** no bureau, e **maior taxa de recusa** em pedidos
> anteriores. Confirma visualmente que o histórico carrega **sinal de risco** — justificando a sua inclusão.

## Conclusões da visão da ABT
- **Dimensão final:** 307.511 linhas × 113 colunas (111 features + `SK_ID_CURR` + `TARGET`),
  partindo de ~83 features só com `application_train`.
- **Tudo numérico:** categóricas já foram codificadas (Label Encoding) e os nulos imputados —
  a tabela entra direto nos modelos do Scikit-Learn.
- **Histórico de crédito:** as 28 features `BUREAU_*`/`PREV_*` adicionam a visão externa (birô) e
  interna (Home Credit). Clientes sem histórico aparecem com 0 (sem registro).
- **Sinal de negócio:** features de **atraso/dívida** (bureau) e **taxa de recusa** (previous)
  tendem a se correlacionar positivamente com a inadimplência; comportamento coerente com o domínio.
- Esta ABT é a entrada do treino (`Model/train.py`); a avaliação do modelo está em
  `Model/evaluation.ipynb`.